# ESJD heatmaps on the $(\rho, \tau)$ grid — uncoupled pCN

Reads the latest pkl produced by `exp_pcn_gaussian.py` and plots, for each SMC
iteration, the (Rao-Blackwellised) ESJD criteria over the 2d $(\rho, \tau)$ grid.
The white curve is the pCN frontier $\tau^2 + \rho^2 = 1$ (on it the proposal
preserves $N(\mu, C)$); the cross marks the $(\rho, \tau)$ selected by the
adaptive procedure at that iteration. Grid points masked by the ESS criterion
(`min_ess_fraction_criteria`) appear in grey.

In [ ]:
import glob
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np

OUTPUT_DIR = "./output/"

pkls = sorted(p for p in glob.glob(os.path.join(OUTPUT_DIR, "*.pkl")) if not p.endswith("_small"))
pkl_path = pkls[-1]  # latest run; set manually to read another one
print("reading", pkl_path)

with open(pkl_path, "rb") as f:
    data = pickle.load(f)
config, res = data["config"], data["res"]

In [ ]:
# res = (couple_particles, log_weights_couple, log_weights, mh_proposal_parameters,
#        acceptance_bools, criteria, tempering_sequence, diff_tempering_sequence,
#        log_normalizations, others), each vmapped over the parallel repetitions.
mh_params = np.asarray(res[3])            # (reps, T + 1, 2)
criteria = np.asarray(res[5])             # (reps, T + 1, n_rho * n_tau)
tempering = np.asarray(res[6])            # (reps, T)

rho_grid = np.asarray(config["rho_grid"])
tau_grid = np.asarray(config["tau_grid"])
n_rho, n_tau = len(rho_grid), len(tau_grid)
n_reps, n_rows, _ = criteria.shape

# The grid was built as [[r, t] for r in rho_grid for t in tau_grid]:
# axis 0 of the reshaped array is rho, axis 1 is tau.
criteria = criteria.reshape(n_reps, n_rows, n_rho, n_tau)
criteria = np.where(np.isfinite(criteria), criteria, np.nan)  # ESS-masked points -> NaN
mean_criteria = np.nanmean(criteria, axis=0)  # average over repetitions

print(f"{n_reps} repetitions, {n_rows} criteria rows, grid {n_rho} x {n_tau}")
print("temperatures:", np.round(tempering[0], 3))

In [ ]:
# Criteria row i is computed at iteration i (temperature lambda_i) and selects the
# (rho, tau) stored in mh_params[:, i], which drives the MH kernel of iteration i + 1.
rho_frontier = np.linspace(0, 1, 200)
tau_frontier = np.sqrt(1 - rho_frontier ** 2)

n_cols = 4
n_plot_rows = int(np.ceil(n_rows / n_cols))
fig, axes = plt.subplots(n_plot_rows, n_cols, figsize=(4 * n_cols, 3.2 * n_plot_rows),
                         sharex=True, sharey=True, constrained_layout=True)

cmap = plt.cm.viridis.copy()
cmap.set_bad("lightgrey")  # ESS-masked grid points

for i, ax in enumerate(np.ravel(axes)):
    if i >= n_rows:
        ax.axis("off")
        continue
    Z = mean_criteria[i]  # (n_rho, n_tau)
    pcm = ax.pcolormesh(rho_grid, tau_grid, Z.T, cmap=cmap, shading="nearest")
    fig.colorbar(pcm, ax=ax)
    ax.plot(rho_frontier, tau_frontier, color="white", lw=1.5, ls="--",
            label=r"$\tau^2 + \rho^2 = 1$")
    for r in range(n_reps):
        ax.plot(*mh_params[r, i], marker="x", color="red", ms=8, mew=2)
    lam = tempering[0, i] if i < tempering.shape[1] else 1.0
    ax.set_title(rf"$t = {i}$, $\lambda_t = {lam:.3f}$")
    ax.set_xlim(rho_grid[0], rho_grid[-1])
    ax.set_ylim(tau_grid[0], tau_grid[-1])

for ax in np.atleast_2d(axes)[-1, :]:
    ax.set_xlabel(r"$\rho$")
for ax in np.atleast_2d(axes)[:, 0]:
    ax.set_ylabel(r"$\tau$")
handles, labels = np.ravel(axes)[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right")
fig.suptitle("ESJD criteria on the $(\\rho, \\tau)$ grid (mean over repetitions); "
             "red cross: selected parameter", y=1.02)
plt.show()

In [ ]:
# Trajectory of the selected (rho, tau) across iterations, against the frontier.
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(rho_frontier, tau_frontier, color="k", lw=1.5, ls="--", label=r"$\tau^2 + \rho^2 = 1$")
for r in range(n_reps):
    sc = ax.scatter(mh_params[r, :, 0], mh_params[r, :, 1], c=np.arange(n_rows),
                    cmap="plasma", s=25, zorder=3)
fig.colorbar(sc, ax=ax, label="iteration $t$")
ax.set_xlabel(r"$\rho$")
ax.set_ylabel(r"$\tau$")
ax.set_xlim(rho_grid[0], rho_grid[-1])
ax.set_ylim(tau_grid[0], tau_grid[-1])
ax.legend()
ax.set_title(r"Selected $(\rho, \tau)$ across iterations")
plt.show()